# Visual Linear Regression: How Does a Machine Find the Best Line?

Welcome! In this session we will discover linear regression the way you would if no one told you the formulas first: by looking at data, drawing lines, and figuring out how to tell a good line from a bad one.

By the end, you will know exactly what `model.fit()` is doing inside scikit-learn — because you will have done it yourself, by hand, first.

### The opening question

> We have information about how many hours students studied and their exam scores. If another student studies **6.5 hours**, how could we estimate their score?

Keep this question in mind. We will answer it properly by the end of Section 15 — and we will not need any statistics jargon to get there.

## Section 1 — Start With Data

Let's look at real(ish) numbers before we talk about models at all.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

pd.set_option("display.precision", 2)

In [2]:
scores_1d = pd.read_csv("../data/study_scores_1d.csv")
scores_1d

,hours_studied,exam_score
0,1.8,47.1
1,2.0,40.2
2,2.8,55.7
3,4.0,56.7
4,4.5,58.6
5,4.5,55.7
6,4.6,67.6
7,6.2,68.2
8,6.6,68.7
9,7.1,71.9


<details>
<summary>Show code</summary>

```python
scatter_figure = go.Figure()

scatter_figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"],
        y=scores_1d["exam_score"],
        mode="markers",
        marker=dict(size=11, color="#2563eb"),
        name="students",
        hovertemplate="Hours studied: %{x}<br>Exam score: %{y}<extra></extra>",
    )
)

scatter_figure.update_layout(
    title="Hours Studied vs Exam Score",
    xaxis_title="Hours studied",
    yaxis_title="Exam score",
    template="plotly_white",
    width=750,
    height=500,
)

scatter_figure.show()
```

</details>

In [3]:
scatter_figure = go.Figure()

scatter_figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"],
        y=scores_1d["exam_score"],
        mode="markers",
        marker=dict(size=11, color="#2563eb"),
        name="students",
        hovertemplate="Hours studied: %{x}<br>Exam score: %{y}<extra></extra>",
    )
)

scatter_figure.update_layout(
    title="Hours Studied vs Exam Score",
    xaxis_title="Hours studied",
    yaxis_title="Exam score",
    template="plotly_white",
    width=750,
    height=500,
)

scatter_figure.show()

> ### 🙋 Ask the class
>
> - Do you see a relationship?
> - If someone studies more, what generally seems to happen?
> - Could one line summarize this relationship?

Before we fit anything, it's worth looking at `hours_studied` on its own — how spread out is it? Are most students clustered around a similar number of hours, or all over the place?

<details>
<summary>Show code</summary>

```python
histogram_figure = go.Figure()
histogram_figure.add_trace(
    go.Histogram(x=scores_1d["hours_studied"], marker=dict(color="#2563eb"))
)
histogram_figure.update_layout(
    title="Distribution of Hours Studied",
    xaxis_title="Hours studied",
    yaxis_title="Number of students",
    template="plotly_white",
    width=750,
    height=400,
)
histogram_figure.show()
```

</details>

In [4]:
histogram_figure = go.Figure()
histogram_figure.add_trace(
    go.Histogram(x=scores_1d["hours_studied"], marker=dict(color="#2563eb"))
)
histogram_figure.update_layout(
    title="Distribution of Hours Studied",
    xaxis_title="Hours studied",
    yaxis_title="Number of students",
    template="plotly_white",
    width=750,
    height=400,
)
histogram_figure.show()

> ### 🙋 Ask the class
>
> - Are study hours spread evenly, or clustered somewhere?

## Section 2 — Human Regression

This is one of the most important parts of the lesson. Before we compute anything, let's just **try lines by hand**.

Our line has the form:

$$\hat y = mx + b$$

In plain language:

```text
m = how steep the line is   (the slope)
b = where the line starts   (the intercept, the value of y when x = 0)
```

Below, `slope` and `intercept` control the line drawn on top of the data. Change the numbers and rerun the cell — or, better, use the Streamlit app (`streamlit run streamlit_app/app.py`, Tab 1) so the plot updates live as you drag sliders.

<details>
<summary>Show code</summary>

```python
def plot_line_over_data(dataframe, slope, intercept, title="Try your own line"):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    line_y = slope * line_x + intercept

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"),
            name="actual students",
            hovertemplate="Hours: %{x}<br>Score: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=line_y, mode="lines",
            line=dict(color="#dc2626", width=3),
            name=f"y = {slope:.1f}x + {intercept:.1f}",
        )
    )
    figure.update_layout(
        title=title,
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure
```

</details>

In [5]:
def plot_line_over_data(dataframe, slope, intercept, title="Try your own line"):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    line_y = slope * line_x + intercept

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"),
            name="actual students",
            hovertemplate="Hours: %{x}<br>Score: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=line_y, mode="lines",
            line=dict(color="#dc2626", width=3),
            name=f"y = {slope:.1f}x + {intercept:.1f}",
        )
    )
    figure.update_layout(
        title=title,
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure

In [6]:
# Try changing these two numbers and rerunning this cell.
slope = 5.0
intercept = 35.0

plot_line_over_data(scores_1d, slope, intercept).show()

> ### 🧑‍🏫 Instructor note
>
> Suggested ranges to keep the exploration sensible for this dataset:
> slope m from -2 to 12, intercept b from 0 to 70.
> Encourage students to shout out numbers and try them live before moving on.

## Section 3 — Manual Line-Fitting Challenge

> **Your challenge:** find a line that visually represents the points as well as possible.

Use only your eyes for now — no error numbers yet. Try the deliberately bad examples below first.

In [7]:
plot_line_over_data(scores_1d, slope=0, intercept=60, title="m = 0, b = 60").show()

> ### 🙋 Ask the class
>
> - What's wrong with this line?

In [8]:
plot_line_over_data(scores_1d, slope=10, intercept=10, title="m = 10, b = 10").show()

> ### 🙋 Ask the class
>
> - What's wrong with this one? Is it wrong in the same way as before?

Now try to find something that looks *reasonable* — a line that seems to pass through the "middle" of the cloud of points.

In [9]:
# Your turn: pick numbers you think look good.
my_slope = 6.0
my_intercept = 30.0

plot_line_over_data(scores_1d, my_slope, my_intercept, title="My attempt").show()

> ### 🧑‍🏫 Instructor note
>
> Different students will land on different (m, b) pairs, and that's the point.
> 
> > Different people may choose slightly different lines. We need an objective way to decide which one is actually better.
> 
> This naturally motivates the need for an error measurement — which is exactly Section 4 and 5.

## Section 4 — Predictions

Let's pick one concrete candidate line and use it to make a prediction for a single student.

$$\hat y = 5x + 35$$

$\hat y$ (read "y hat") is just notation for **the model's prediction** — as opposed to $y$, the actual observed score.

<details>
<summary>Show code</summary>

```python
candidate_slope = 5.0
candidate_intercept = 35.0

scores_1d["predicted_score"] = candidate_slope * scores_1d["hours_studied"] + candidate_intercept

example_row = scores_1d.iloc[5]

figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="One prediction, highlighted")
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["exam_score"]],
        mode="markers", marker=dict(size=16, color="#16a34a", symbol="circle"),
        name="Actual score",
    )
)
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["predicted_score"]],
        mode="markers", marker=dict(size=16, color="#dc2626", symbol="x"),
        name="Predicted score",
    )
)
figure.show()

print(f"Student studied {example_row['hours_studied']} hours.")
print(f"Actual score:    {example_row['exam_score']:.1f}")
print(f"Predicted score: {example_row['predicted_score']:.1f}")
```

</details>

In [10]:
candidate_slope = 5.0
candidate_intercept = 35.0

scores_1d["predicted_score"] = candidate_slope * scores_1d["hours_studied"] + candidate_intercept

example_row = scores_1d.iloc[5]

figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="One prediction, highlighted")
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["exam_score"]],
        mode="markers", marker=dict(size=16, color="#16a34a", symbol="circle"),
        name="Actual score",
    )
)
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["predicted_score"]],
        mode="markers", marker=dict(size=16, color="#dc2626", symbol="x"),
        name="Predicted score",
    )
)
figure.show()

print(f"Student studied {example_row['hours_studied']} hours.")
print(f"Actual score:    {example_row['exam_score']:.1f}")
print(f"Predicted score: {example_row['predicted_score']:.1f}")

Student studied 4.5 hours.
Actual score:    55.7
Predicted score: 57.5


## Section 5 — The Error

For every student, there's a gap between what actually happened and what the line predicted. Draw that gap as a vertical segment.

$$e_i = y_i - \hat y_i$$

<details>
<summary>Show code</summary>

```python
figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="Errors as vertical segments")

for _, row in scores_1d.iterrows():
    figure.add_trace(
        go.Scatter(
            x=[row["hours_studied"], row["hours_studied"]],
            y=[row["predicted_score"], row["exam_score"]],
            mode="lines",
            line=dict(color="#f59e0b", width=2, dash="dot"),
            showlegend=False,
        )
    )

figure.show()
```

</details>

In [11]:
figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="Errors as vertical segments")

for _, row in scores_1d.iterrows():
    figure.add_trace(
        go.Scatter(
            x=[row["hours_studied"], row["hours_studied"]],
            y=[row["predicted_score"], row["exam_score"]],
            mode="lines",
            line=dict(color="#f59e0b", width=2, dash="dot"),
            showlegend=False,
        )
    )

figure.show()

In [12]:
error_table = scores_1d.copy()
error_table["error"] = error_table["exam_score"] - error_table["predicted_score"]
error_table = error_table.rename(columns={"hours_studied": "Hours", "exam_score": "Actual", "predicted_score": "Prediction", "error": "Error"})
error_table[["Hours", "Actual", "Prediction", "Error"]].round(2)

,Hours,Actual,Prediction,Error
0,1.8,47.1,44.0,3.1
1,2.0,40.2,45.0,-4.8
2,2.8,55.7,49.0,6.7
3,4.0,56.7,55.0,1.7
4,4.5,58.6,57.5,1.1
5,4.5,55.7,57.5,-1.8
6,4.6,67.6,58.0,9.6
7,6.2,68.2,66.0,2.2
8,6.6,68.7,68.0,0.7
9,7.1,71.9,70.5,1.4


> ### 🙋 Ask the class
>
> - If we want one number describing how bad the entire line is, why can't we just add all these errors?

In [13]:
# A tiny illustration of cancellation.
error_a = 4
error_b = -4

print("error_a + error_b =", error_a + error_b)
print("But both predictions were wrong!")

error_a + error_b = 0
But both predictions were wrong!


> ### 🧑‍🏫 Instructor note
>
> Positive and negative errors cancel when summed directly, so the total can look like a perfect fit (sum = 0)
> even when every single prediction was wrong. We need a way to make all errors count as "bad," regardless of sign.

## Section 6 — Squaring the Error

$$e_i^2 = (y_i - \hat y_i)^2$$

Squaring solves the cancellation problem, and gives us two more useful properties:

1. Negative errors become positive.
2. Errors cannot cancel each other out.
3. Large mistakes receive much larger penalties than small ones.

In [14]:
for error in [2, 5, 10]:
    print(f"error = {error:>2} -> squared error = {error ** 2}")

error =  2 -> squared error = 4
error =  5 -> squared error = 25
error = 10 -> squared error = 100


In [15]:
error_table["Squared Error"] = error_table["Error"] ** 2
error_table[["Hours", "Actual", "Prediction", "Error", "Squared Error"]].round(2)

,Hours,Actual,Prediction,Error,Squared Error
0,1.8,47.1,44.0,3.1,9.61
1,2.0,40.2,45.0,-4.8,23.04
2,2.8,55.7,49.0,6.7,44.89
3,4.0,56.7,55.0,1.7,2.89
4,4.5,58.6,57.5,1.1,1.21
5,4.5,55.7,57.5,-1.8,3.24
6,4.6,67.6,58.0,9.6,92.16
7,6.2,68.2,66.0,2.2,4.84
8,6.6,68.7,68.0,0.7,0.49
9,7.1,71.9,70.5,1.4,1.96


> This also means least squares can be sensitive to outliers — one very wrong point contributes a disproportionately large squared error. (We won't cover robust regression today, just keep this in the back of your mind.)

## Section 7 — Sum of Squared Errors (SSE)

$$SSE = \sum_{i=1}^{n} (y_i - \hat y_i)^2$$

We now have **one number** representing how badly the line fits all observations.

```text
large SSE  = poor fit
smaller SSE = better fit
```

<details>
<summary>Show code</summary>

```python
def compute_sse(dataframe, slope, intercept):
    predictions = slope * dataframe["hours_studied"] + intercept
    residuals = dataframe["exam_score"] - predictions
    squared_errors = residuals ** 2
    sse = squared_errors.sum()
    return sse

current_sse = compute_sse(scores_1d, candidate_slope, candidate_intercept)
print(f"Current line: y = {candidate_slope}x + {candidate_intercept}")
print(f"Current SSE: {current_sse:.1f}")
```

</details>

In [16]:
def compute_sse(dataframe, slope, intercept):
    predictions = slope * dataframe["hours_studied"] + intercept
    residuals = dataframe["exam_score"] - predictions
    squared_errors = residuals ** 2
    sse = squared_errors.sum()
    return sse

current_sse = compute_sse(scores_1d, candidate_slope, candidate_intercept)
print(f"Current line: y = {candidate_slope}x + {candidate_intercept}")
print(f"Current SSE: {current_sse:.1f}")

Current line: y = 5.0x + 35.0
Current SSE: 519.1


In [5]:
figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept,
                              title=f"y = {candidate_slope}x + {candidate_intercept}   |   SSE = {current_sse:.1f}")
figure.show()

NameError: name 'plot_line_over_data' is not defined

## Section 8 — The Regression Game

Now it's a competition. Try different `slope` / `intercept` pairs below and watch the SSE. **You are training the regression model manually.**

> Can anyone get SSE below 500? Below 300? Below 200?

For the full live-slider experience, switch to the Streamlit app now: **Tab 1 — Fit a Line**.

In [18]:
# Change these and rerun.
game_slope = 5.5
game_intercept = 35.0

game_sse = compute_sse(scores_1d, game_slope, game_intercept)
figure = plot_line_over_data(scores_1d, game_slope, game_intercept,
                              title=f"y = {game_slope}x + {game_intercept}   |   SSE = {game_sse:.1f}")
figure.show()
print(f"Current SSE: {game_sse:.1f}")

Current SSE: 260.7


> ### 🧑‍🏫 Instructor note
>
> Adjust the "below X" thresholds to the actual dataset if you regenerate the data — a reasonable target is
> roughly 20-40% above the true minimum SSE (computed later in Section 13). Don't reveal the optimal line yet.

## Section 9 — Visualizing Residuals

Let's build one clean, reusable visualization: points, the candidate line, and every residual as a vertical segment, with rich hover info. We'll reuse this function for the rest of the notebook.

<details>
<summary>Show code</summary>

```python
def plot_line_with_residuals(dataframe, slope, intercept, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predictions = slope * x + intercept
    residuals = y - predictions
    squared_residuals = residuals ** 2

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    line_y = slope * line_x + intercept

    figure = go.Figure()

    for xi, yi, pi, ri, sqi in zip(x, y, predictions, residuals, squared_residuals):
        figure.add_trace(
            go.Scatter(
                x=[xi, xi], y=[pi, yi], mode="lines",
                line=dict(color="#f59e0b", width=2, dash="dot"),
                showlegend=False,
                hovertemplate=(
                    f"x: {xi:.1f}<br>actual: {yi:.1f}<br>prediction: {pi:.1f}"
                    f"<br>residual: {ri:.1f}<br>squared residual: {sqi:.1f}<extra></extra>"
                ),
            )
        )

    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"),
            name="actual",
            hovertemplate="x: %{x}<br>actual: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=slope * line_x + intercept, mode="lines",
            line=dict(color="#dc2626", width=3),
            name=f"y = {slope:.2f}x + {intercept:.2f}",
        )
    )

    sse = float((residuals ** 2).sum())
    figure.update_layout(
        title=title or f"SSE = {sse:.1f}",
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure, sse

figure, sse = plot_line_with_residuals(scores_1d, candidate_slope, candidate_intercept)
figure.show()
```

</details>

In [19]:
def plot_line_with_residuals(dataframe, slope, intercept, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predictions = slope * x + intercept
    residuals = y - predictions
    squared_residuals = residuals ** 2

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    line_y = slope * line_x + intercept

    figure = go.Figure()

    for xi, yi, pi, ri, sqi in zip(x, y, predictions, residuals, squared_residuals):
        figure.add_trace(
            go.Scatter(
                x=[xi, xi], y=[pi, yi], mode="lines",
                line=dict(color="#f59e0b", width=2, dash="dot"),
                showlegend=False,
                hovertemplate=(
                    f"x: {xi:.1f}<br>actual: {yi:.1f}<br>prediction: {pi:.1f}"
                    f"<br>residual: {ri:.1f}<br>squared residual: {sqi:.1f}<extra></extra>"
                ),
            )
        )

    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"),
            name="actual",
            hovertemplate="x: %{x}<br>actual: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=slope * line_x + intercept, mode="lines",
            line=dict(color="#dc2626", width=3),
            name=f"y = {slope:.2f}x + {intercept:.2f}",
        )
    )

    sse = float((residuals ** 2).sum())
    figure.update_layout(
        title=title or f"SSE = {sse:.1f}",
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure, sse

figure, sse = plot_line_with_residuals(scores_1d, candidate_slope, candidate_intercept)
figure.show()

So far every point on our charts has been a **training observation** — a real student with a real, known score. But the entire reason we fit a line is to answer questions about students we *haven't* seen yet. Let's build one more reusable visualization for exactly that: reading a prediction for a brand-new x value off the chart.

<details>
<summary>Show code</summary>

```python
def plot_new_prediction(dataframe, slope, intercept, new_x, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predicted_y = slope * new_x + intercept

    line_x = np.linspace(min(x.min(), new_x) - 0.5, max(x.max(), new_x) + 0.5, 50)

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"), name="training students",
            hovertemplate="Hours: %{x}<br>Actual: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=slope * line_x + intercept, mode="lines",
            line=dict(color="#dc2626", width=3), name=f"y = {slope:.2f}x + {intercept:.2f}",
        )
    )

    # dashed guide lines: up from the x-axis to the line, then across to the y-axis
    figure.add_trace(
        go.Scatter(
            x=[new_x, new_x], y=[0, predicted_y], mode="lines",
            line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[0, new_x], y=[predicted_y, predicted_y], mode="lines",
            line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[new_x], y=[predicted_y], mode="markers",
            marker=dict(size=16, color="#7c3aed", symbol="star"),
            name="new prediction (no actual value)",
            hovertemplate=f"x: {new_x} (new / unseen)<br>predicted y: {predicted_y:.1f}<extra></extra>",
        )
    )

    figure.update_layout(
        title=title or f"Predicting for an unseen x = {new_x}",
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure, predicted_y

demo_figure, demo_prediction = plot_new_prediction(scores_1d, candidate_slope, candidate_intercept, new_x=6.5)
demo_figure.show()
```

</details>

In [20]:
def plot_new_prediction(dataframe, slope, intercept, new_x, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predicted_y = slope * new_x + intercept

    line_x = np.linspace(min(x.min(), new_x) - 0.5, max(x.max(), new_x) + 0.5, 50)

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers",
            marker=dict(size=11, color="#2563eb"), name="training students",
            hovertemplate="Hours: %{x}<br>Actual: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=slope * line_x + intercept, mode="lines",
            line=dict(color="#dc2626", width=3), name=f"y = {slope:.2f}x + {intercept:.2f}",
        )
    )

    # dashed guide lines: up from the x-axis to the line, then across to the y-axis
    figure.add_trace(
        go.Scatter(
            x=[new_x, new_x], y=[0, predicted_y], mode="lines",
            line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[0, new_x], y=[predicted_y, predicted_y], mode="lines",
            line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False,
        )
    )
    figure.add_trace(
        go.Scatter(
            x=[new_x], y=[predicted_y], mode="markers",
            marker=dict(size=16, color="#7c3aed", symbol="star"),
            name="new prediction (no actual value)",
            hovertemplate=f"x: {new_x} (new / unseen)<br>predicted y: {predicted_y:.1f}<extra></extra>",
        )
    )

    figure.update_layout(
        title=title or f"Predicting for an unseen x = {new_x}",
        xaxis_title="Hours studied",
        yaxis_title="Exam score",
        template="plotly_white",
        width=750,
        height=500,
    )
    return figure, predicted_y

demo_figure, demo_prediction = plot_new_prediction(scores_1d, candidate_slope, candidate_intercept, new_x=6.5)
demo_figure.show()

## Section 10 — What Are We Actually Searching For?

Let's formalize what we've been doing. Our model:

$$\hat y = mx + b$$

Our loss (how bad a given line is):

$$L(m, b) = \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

The **dataset is fixed** — we cannot change the data. The only things we can change are:

```text
m   (slope)
b   (intercept)
```

So regression is really asking one question:

> Which values of m and b produce the smallest error?

```text
(m, b)
   ↓
predictions
   ↓
residuals
   ↓
SSE
```

> ### 🙋 Ask the class
>
> - What exactly are we changing when we "train" this model?

## Section 11 — Visualize the Loss Landscape

If SSE depends on `m` and `b`, we can plot SSE as a **surface** over every possible (m, b) pair. This turns training into something you can literally see: finding the bottom of a valley.

<details>
<summary>Show code</summary>

```python
slope_range = np.linspace(-2, 12, 60)
intercept_range = np.linspace(0, 70, 60)
slope_grid, intercept_grid = np.meshgrid(slope_range, intercept_range)

sse_grid = np.zeros_like(slope_grid)
x_values = scores_1d["hours_studied"].to_numpy()
y_values = scores_1d["exam_score"].to_numpy()

for row in range(slope_grid.shape[0]):
    for col in range(slope_grid.shape[1]):
        m = slope_grid[row, col]
        b = intercept_grid[row, col]
        predictions = m * x_values + b
        sse_grid[row, col] = np.sum((y_values - predictions) ** 2)

best_index = np.unravel_index(np.argmin(sse_grid), sse_grid.shape)
best_slope = slope_grid[best_index]
best_intercept = intercept_grid[best_index]
best_sse = sse_grid[best_index]

surface_figure = go.Figure(
    data=[go.Surface(x=slope_grid, y=intercept_grid, z=sse_grid, colorscale="Viridis", opacity=0.9)]
)
surface_figure.add_trace(
    go.Scatter3d(
        x=[best_slope], y=[best_intercept], z=[best_sse],
        mode="markers", marker=dict(size=6, color="red"),
        name="minimum",
    )
)
surface_figure.update_layout(
    title="SSE Loss Landscape",
    scene=dict(xaxis_title="slope (m)", yaxis_title="intercept (b)", zaxis_title="SSE"),
    width=800, height=600,
)
surface_figure.show()

print(f"Approximate minimum near: slope={best_slope:.2f}, intercept={best_intercept:.2f}, SSE={best_sse:.1f}")
```

</details>

In [21]:
slope_range = np.linspace(-2, 12, 60)
intercept_range = np.linspace(0, 70, 60)
slope_grid, intercept_grid = np.meshgrid(slope_range, intercept_range)

sse_grid = np.zeros_like(slope_grid)
x_values = scores_1d["hours_studied"].to_numpy()
y_values = scores_1d["exam_score"].to_numpy()

for row in range(slope_grid.shape[0]):
    for col in range(slope_grid.shape[1]):
        m = slope_grid[row, col]
        b = intercept_grid[row, col]
        predictions = m * x_values + b
        sse_grid[row, col] = np.sum((y_values - predictions) ** 2)

best_index = np.unravel_index(np.argmin(sse_grid), sse_grid.shape)
best_slope = slope_grid[best_index]
best_intercept = intercept_grid[best_index]
best_sse = sse_grid[best_index]

surface_figure = go.Figure(
    data=[go.Surface(x=slope_grid, y=intercept_grid, z=sse_grid, colorscale="Viridis", opacity=0.9)]
)
surface_figure.add_trace(
    go.Scatter3d(
        x=[best_slope], y=[best_intercept], z=[best_sse],
        mode="markers", marker=dict(size=6, color="red"),
        name="minimum",
    )
)
surface_figure.update_layout(
    title="SSE Loss Landscape",
    scene=dict(xaxis_title="slope (m)", yaxis_title="intercept (b)", zaxis_title="SSE"),
    width=800, height=600,
)
surface_figure.show()

print(f"Approximate minimum near: slope={best_slope:.2f}, intercept={best_intercept:.2f}, SSE={best_sse:.1f}")

Approximate minimum near: slope=5.83, intercept=34.41, SSE=241.3


> ### 🙋 Ask the class
>
> - What would "training" mean on this surface?

> ### 🧑‍🏫 Instructor note
>
> Expected answer: find the lowest point.
> 
> Important statement to make out loud:
> Machine learning frequently turns into an optimization problem: choose parameters that minimize some loss function.
> Consider hiding the red minimum marker on first display and revealing it only after discussion, if presenting live.

## Section 12 — Connection to Calculus

Keep this section light — no long derivation. At the very bottom of the valley, the surface is momentarily flat in every direction. That means the slope of the loss surface itself is zero there:

$$\frac{\partial L}{\partial m} = 0 \qquad \frac{\partial L}{\partial b} = 0$$

```text
derivatives
   ↓
minimum
   ↓
model parameters
```

This is the same derivative intuition you already have — "the derivative is zero at a minimum" — now applied to a machine learning loss function instead of a textbook function.

## Section 13 — Calculate the Best Line Manually

For simple (one-variable) linear regression, there's a direct formula for the exact minimum — no searching required:

$$m = \frac{\sum (x_i - \bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2} \qquad b = \bar y - m \bar x$$

> Students should understand what these formulas **find**. They do not need to memorize them.

Let's implement this in transparent steps.

<details>
<summary>Show code</summary>

```python
x_mean = scores_1d["hours_studied"].mean()
y_mean = scores_1d["exam_score"].mean()

deviations_x = scores_1d["hours_studied"] - x_mean
deviations_y = scores_1d["exam_score"] - y_mean

numerator = (deviations_x * deviations_y).sum()
denominator = (deviations_x ** 2).sum()

manual_slope = numerator / denominator
manual_intercept = y_mean - manual_slope * x_mean

print(f"x_mean:    {x_mean:.3f}")
print(f"y_mean:    {y_mean:.3f}")
print(f"numerator:   {numerator:.3f}")
print(f"denominator: {denominator:.3f}")
print()
print(f"slope (m):     {manual_slope:.4f}")
print(f"intercept (b): {manual_intercept:.4f}")
```

</details>

In [22]:
x_mean = scores_1d["hours_studied"].mean()
y_mean = scores_1d["exam_score"].mean()

deviations_x = scores_1d["hours_studied"] - x_mean
deviations_y = scores_1d["exam_score"] - y_mean

numerator = (deviations_x * deviations_y).sum()
denominator = (deviations_x ** 2).sum()

manual_slope = numerator / denominator
manual_intercept = y_mean - manual_slope * x_mean

print(f"x_mean:    {x_mean:.3f}")
print(f"y_mean:    {y_mean:.3f}")
print(f"numerator:   {numerator:.3f}")
print(f"denominator: {denominator:.3f}")
print()
print(f"slope (m):     {manual_slope:.4f}")
print(f"intercept (b): {manual_intercept:.4f}")

x_mean:    5.706
y_mean:    67.300
numerator:   451.340
denominator: 77.469

slope (m):     5.8260
intercept (b): 34.0551


In [23]:
manual_sse = compute_sse(scores_1d, manual_slope, manual_intercept)
print(f"SSE with manual least-squares line: {manual_sse:.2f}")
print(f"SSE with your earlier manual guess ({game_slope}, {game_intercept}): {game_sse:.2f}")

figure, _ = plot_line_with_residuals(scores_1d, manual_slope, manual_intercept,
                                      title=f"Manual least-squares line | SSE = {manual_sse:.1f}")
figure.show()

SSE with manual least-squares line: 239.03
SSE with your earlier manual guess (5.5, 35.0): 260.68


> ### 🧑‍🏫 Instructor note
>
> Compare this manual_sse directly against whatever SSE the class achieved in the Section 8 game.
> The formula-derived line should beat (or tie) every hand-picked line.

## Section 14 — Finally Use Scikit-Learn

Only now do we bring in a library. Notice: everything sklearn is about to do, we already did by hand above.

In [24]:
from sklearn.linear_model import LinearRegression

X = scores_1d[["hours_studied"]]   # shape (n_samples, 1) -- sklearn always wants a 2D X
y = scores_1d["exam_score"]        # shape (n_samples,)

print("X.shape:", X.shape)
print("y.shape:", y.shape)

X.shape: (16, 1)
y.shape: (16,)


In [25]:
model = LinearRegression()
model.fit(X, y)

print("model.coef_:     ", model.coef_)
print("model.intercept_:", model.intercept_)

model.coef_:      [5.82604416]
model.intercept_: 34.055135497091605


<details>
<summary>Show code</summary>

```python
sklearn_slope = model.coef_[0]
sklearn_intercept = model.intercept_
sklearn_sse = compute_sse(scores_1d, sklearn_slope, sklearn_intercept)

comparison = pd.DataFrame({
    "Method": ["Human attempt", "Manual least squares", "sklearn"],
    "Slope": [game_slope, manual_slope, sklearn_slope],
    "Intercept": [game_intercept, manual_intercept, sklearn_intercept],
    "SSE": [game_sse, manual_sse, sklearn_sse],
})
comparison.round(4)
```

</details>

In [26]:
sklearn_slope = model.coef_[0]
sklearn_intercept = model.intercept_
sklearn_sse = compute_sse(scores_1d, sklearn_slope, sklearn_intercept)

comparison = pd.DataFrame({
    "Method": ["Human attempt", "Manual least squares", "sklearn"],
    "Slope": [game_slope, manual_slope, sklearn_slope],
    "Intercept": [game_intercept, manual_intercept, sklearn_intercept],
    "SSE": [game_sse, manual_sse, sklearn_sse],
})
comparison.round(4)

,Method,Slope,Intercept,SSE
0,Human attempt,5.50,35.00,260.68
1,Manual least squares,5.83,34.06,239.03
2,sklearn,5.83,34.06,239.03


The **manual least-squares** and **sklearn** rows should match to numerical precision. This is the "aha" moment.

> `model.fit()` looked like one line of code. But you now know exactly what mathematical problem that one line solved.

## Section 15 — Prediction

Back to our opening question: a new student studies **6.5 hours**. What score do we predict?

First, by hand:

In [27]:
new_hours = 6.5
manual_prediction = sklearn_slope * new_hours + sklearn_intercept
print(f"Manual prediction: {manual_prediction:.2f}")

Manual prediction: 71.92


In [28]:
sklearn_prediction = model.predict([[new_hours]])
print(f"sklearn prediction: {sklearn_prediction[0]:.2f}")

sklearn prediction: 71.92


/Users/abdukarimov/workspaces/work/humblebeeai/int.academy-tutorial/linear-regression/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Let's see it, not just read it. The point below sits **on the fitted line** at x = 6.5, but unlike every blue dot around it, it has no real observed score — it's purely a model output.

In [29]:
prediction_figure, _ = plot_new_prediction(scores_1d, sklearn_slope, sklearn_intercept, new_x=new_hours,
                                            title=f"Predicting for a new student who studied {new_hours} hours")
prediction_figure.show()

Same answer, two routes. This distinction matters:

```text
Training:   finding parameters   (model.fit)
Prediction: using those parameters (model.predict)
```

## Section 16 — Your Turn: One-Variable Practice

We'll reuse the same dataset. Work through the steps below on your own before opening the solutions.

1. Plot the data.
2. Guess a slope and intercept just by looking.
3. Calculate predictions for every point.
4. Calculate residuals.
5. Calculate squared residuals.
6. Calculate SSE.
7. Fit using sklearn.
8. Compare your SSE with sklearn's SSE.
9. Predict the score for a student who studies 4 hours.

In [30]:
# 1-2. Plot the data and pick your own slope/intercept.
practice_slope = ...   # replace with a number
practice_intercept = ...   # replace with a number

<details>
<summary>Show solution</summary>

```python
practice_slope = 5.5
practice_intercept = 33.0
plot_line_over_data(scores_1d, practice_slope, practice_intercept).show()
```

</details>

In [31]:
# 3-6. Predictions, residuals, squared residuals, SSE.

<details>
<summary>Show solution</summary>

```python
practice_predictions = practice_slope * scores_1d["hours_studied"] + practice_intercept
practice_residuals = scores_1d["exam_score"] - practice_predictions
practice_squared_errors = practice_residuals ** 2
practice_sse = practice_squared_errors.sum()
print("Your SSE:", practice_sse)
```

</details>

In [32]:
# 7-8. Fit with sklearn and compare.

<details>
<summary>Show solution</summary>

```python
practice_model = LinearRegression()
practice_model.fit(scores_1d[["hours_studied"]], scores_1d["exam_score"])
practice_sklearn_sse = compute_sse(scores_1d, practice_model.coef_[0], practice_model.intercept_)

print("Your SSE:     ", practice_sse)
print("sklearn SSE:  ", practice_sklearn_sse)
```

</details>

In [33]:
# 9. Predict for 4 hours studied.

<details>
<summary>Show solution</summary>

```python
print(practice_model.predict([[4]]))
```

</details>

## What If One Point Is Way Off?

Back in Section 6 we mentioned that squaring errors means least squares can be sensitive to outliers. Let's actually see it.

Imagine one more student is added to our dataset: they studied for just **1 hour** but somehow scored **95**. Everyone else still looks the same.

<details>
<summary>Show code</summary>

```python
outlier_row = pd.DataFrame([{"hours_studied": 1.0, "exam_score": 95.0}])
scores_1d_with_outlier = pd.concat([scores_1d, outlier_row], ignore_index=True)

outlier_model = LinearRegression()
outlier_model.fit(scores_1d_with_outlier[["hours_studied"]], scores_1d_with_outlier["exam_score"])
outlier_slope = outlier_model.coef_[0]
outlier_intercept = outlier_model.intercept_

line_x = np.linspace(scores_1d["hours_studied"].min() - 0.5, scores_1d["hours_studied"].max() + 0.5, 50)

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="original students",
    )
)
figure.add_trace(
    go.Scatter(
        x=outlier_row["hours_studied"], y=outlier_row["exam_score"], mode="markers",
        marker=dict(size=16, color="#dc2626", symbol="diamond"), name="outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=sklearn_slope * line_x + sklearn_intercept, mode="lines",
        line=dict(color="#16a34a", width=3, dash="dash"), name="best fit WITHOUT outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=outlier_slope * line_x + outlier_intercept, mode="lines",
        line=dict(color="#dc2626", width=3), name="best fit WITH outlier",
    )
)
figure.update_layout(
    title="One Outlier, Two Very Different Lines",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
figure.show()

comparison_with_outlier = pd.DataFrame({
    "Fit": ["Without outlier", "With outlier"],
    "Slope": [sklearn_slope, outlier_slope],
    "Intercept": [sklearn_intercept, outlier_intercept],
    "SSE (on original 16 points)": [
        compute_sse(scores_1d, sklearn_slope, sklearn_intercept),
        compute_sse(scores_1d, outlier_slope, outlier_intercept),
    ],
})
comparison_with_outlier.round(3)
```

</details>

In [34]:
outlier_row = pd.DataFrame([{"hours_studied": 1.0, "exam_score": 95.0}])
scores_1d_with_outlier = pd.concat([scores_1d, outlier_row], ignore_index=True)

outlier_model = LinearRegression()
outlier_model.fit(scores_1d_with_outlier[["hours_studied"]], scores_1d_with_outlier["exam_score"])
outlier_slope = outlier_model.coef_[0]
outlier_intercept = outlier_model.intercept_

line_x = np.linspace(scores_1d["hours_studied"].min() - 0.5, scores_1d["hours_studied"].max() + 0.5, 50)

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="original students",
    )
)
figure.add_trace(
    go.Scatter(
        x=outlier_row["hours_studied"], y=outlier_row["exam_score"], mode="markers",
        marker=dict(size=16, color="#dc2626", symbol="diamond"), name="outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=sklearn_slope * line_x + sklearn_intercept, mode="lines",
        line=dict(color="#16a34a", width=3, dash="dash"), name="best fit WITHOUT outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=outlier_slope * line_x + outlier_intercept, mode="lines",
        line=dict(color="#dc2626", width=3), name="best fit WITH outlier",
    )
)
figure.update_layout(
    title="One Outlier, Two Very Different Lines",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
figure.show()

comparison_with_outlier = pd.DataFrame({
    "Fit": ["Without outlier", "With outlier"],
    "Slope": [sklearn_slope, outlier_slope],
    "Intercept": [sklearn_intercept, outlier_intercept],
    "SSE (on original 16 points)": [
        compute_sse(scores_1d, sklearn_slope, sklearn_intercept),
        compute_sse(scores_1d, outlier_slope, outlier_intercept),
    ],
})
comparison_with_outlier.round(3)

,Fit,Slope,Intercept,SSE (on original 16 points)
0,Without outlier,5.83,34.05,239.03
1,With outlier,3.34,50.78,821.19


A single point — one out of seventeen — pulled the slope down and the intercept up. The line minimizes squared error across *every* point, so one point with a huge error (before fitting) gets a huge say in where the line ends up.

> This isn't something we'll fix today (that's what robust regression techniques are for), but it's important to know least squares has this soft spot.

## Section 17 — The Big Question

> Real models normally don't only receive one piece of information. What happens if the prediction depends on **two** things?

Let's predict exam score from **both** hours studied **and** number of practice problems solved.

In [35]:
scores_2d = pd.read_csv("../data/study_scores_2d.csv")
scores_2d.head(10)

,hours_studied,practice_problems,exam_score
0,6.0,20,87.7
1,8.2,0,63.1
2,7.2,2,77.4
3,2.8,4,53.2
4,3.4,20,68.1
5,8.0,14,87.4
6,1.0,18,65.0
7,7.6,4,69.1
8,7.4,15,89.9
9,4.7,7,61.3


Our model now has two coefficients, one per input:

$$\hat y = b + w_1 x_1 + w_2 x_2$$

```text
Previously:  one input  -> one coefficient -> a LINE
Now:         two inputs -> two coefficients -> a PLANE
```

## Section 18 — 3D Regression

Let's look at the raw points in 3D first — no plane yet. Rotate the plot and look for a flat surface that would pass roughly through the cloud of points.

<details>
<summary>Show code</summary>

```python
scatter_3d = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"],
        y=scores_2d["practice_problems"],
        z=scores_2d["exam_score"],
        mode="markers",
        marker=dict(size=5, color="#2563eb"),
        hovertemplate="Hours: %{x}<br>Problems: %{y}<br>Score: %{z}<extra></extra>",
    )]
)
scatter_3d.update_layout(
    title="Exam Score vs Hours Studied and Practice Problems",
    scene=dict(
        xaxis_title="Hours studied",
        yaxis_title="Practice problems",
        zaxis_title="Exam score",
    ),
    width=800, height=650,
)
scatter_3d.show()
```

</details>

In [36]:
scatter_3d = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"],
        y=scores_2d["practice_problems"],
        z=scores_2d["exam_score"],
        mode="markers",
        marker=dict(size=5, color="#2563eb"),
        hovertemplate="Hours: %{x}<br>Problems: %{y}<br>Score: %{z}<extra></extra>",
    )]
)
scatter_3d.update_layout(
    title="Exam Score vs Hours Studied and Practice Problems",
    scene=dict(
        xaxis_title="Hours studied",
        yaxis_title="Practice problems",
        zaxis_title="Exam score",
    ),
    width=800, height=650,
)
scatter_3d.show()

> ### 🙋 Ask the class
>
> - Where would you put a flat surface that approximately passes through these points?

<details>
<summary>Show code</summary>

```python
def add_regression_plane(figure, dataframe, intercept, w1, w2, opacity=0.5):
    x_range = np.linspace(dataframe["hours_studied"].min(), dataframe["hours_studied"].max(), 15)
    y_range = np.linspace(dataframe["practice_problems"].min(), dataframe["practice_problems"].max(), 15)
    x_grid, y_grid = np.meshgrid(x_range, y_range)
    z_grid = intercept + w1 * x_grid + w2 * y_grid

    figure.add_trace(
        go.Surface(x=x_grid, y=y_grid, z=z_grid, opacity=opacity, colorscale="Reds", showscale=False, name="plane")
    )
    return figure

plane_2d_model = LinearRegression()
plane_2d_model.fit(scores_2d[["hours_studied", "practice_problems"]], scores_2d["exam_score"])

plane_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"),
    )]
)
add_regression_plane(
    plane_figure, scores_2d,
    plane_2d_model.intercept_, plane_2d_model.coef_[0], plane_2d_model.coef_[1],
)
plane_figure.update_layout(
    title="Points with a Fitted Regression Plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
plane_figure.show()
```

</details>

In [37]:
def add_regression_plane(figure, dataframe, intercept, w1, w2, opacity=0.5):
    x_range = np.linspace(dataframe["hours_studied"].min(), dataframe["hours_studied"].max(), 15)
    y_range = np.linspace(dataframe["practice_problems"].min(), dataframe["practice_problems"].max(), 15)
    x_grid, y_grid = np.meshgrid(x_range, y_range)
    z_grid = intercept + w1 * x_grid + w2 * y_grid

    figure.add_trace(
        go.Surface(x=x_grid, y=y_grid, z=z_grid, opacity=opacity, colorscale="Reds", showscale=False, name="plane")
    )
    return figure

plane_2d_model = LinearRegression()
plane_2d_model.fit(scores_2d[["hours_studied", "practice_problems"]], scores_2d["exam_score"])

plane_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"),
    )]
)
add_regression_plane(
    plane_figure, scores_2d,
    plane_2d_model.intercept_, plane_2d_model.coef_[0], plane_2d_model.coef_[1],
)
plane_figure.update_layout(
    title="Points with a Fitted Regression Plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
plane_figure.show()

## Section 19 — Manual Plane Fitting

Just like we manually adjusted `slope` and `intercept` for a line, we can manually adjust the plane's three numbers:

$$\hat y = b + w_1 x_1 + w_2 x_2$$

```text
Intercept b
Hours coefficient w1
Practice coefficient w2
```

**From here, switch to the Streamlit app** (`streamlit run streamlit_app/app.py`, **Tab 4 — Fit a Plane**) for the full interactive experience — live SSE, rotation, residual lines, and a one-click sklearn reveal.

Core teaching analogy:

```text
1 feature  -> adjust a LINE
2 features -> adjust a PLANE
```

In [38]:
def compute_sse_2d(dataframe, intercept, w1, w2):
    predictions = intercept + w1 * dataframe["hours_studied"] + w2 * dataframe["practice_problems"]
    residuals = dataframe["exam_score"] - predictions
    return (residuals ** 2).sum()

# Try changing these three numbers.
my_intercept = 30.0
my_w1 = 4.0
my_w2 = 1.0

print("Your SSE:", compute_sse_2d(scores_2d, my_intercept, my_w1, my_w2))

Your SSE: 2926.560000000001


> ### 🧑‍🏫 Instructor note
>
> This is the last notebook cell before handing off to Streamlit. Suggested classroom sequence:
> show points only -> ask what a model would look like -> reveal a deliberately bad plane -> let students
> change w1 and w2 one at a time and ask what each one visually controls -> turn on residuals -> minimize
> SSE manually -> reveal the sklearn optimum. See PLAN.md Section 22 for the full script.

## Section 20 — Many Features: How Far Does This Idea Go?

We went from **1 feature** (a line) to **2 features** (a plane). What about 3? Or 40?

```text
1 feature:   ŷ = b + w1*x1                     -> LINE
2 features:  ŷ = b + w1*x1 + w2*x2              -> PLANE
3 features:  ŷ = b + w1*x1 + w2*x2 + w3*x3      -> beyond what we can draw
```

> We are now beyond what we can conveniently visualize, because the output would need a 4th dimension.

The pattern keeps going. With 40 features:

$$\hat y = b + \sum_{j=1}^{40} w_j x_j$$

> ### 🙋 Ask the class
>
> - What do you think replaces the line in three dimensions?
> - What happens if we have 40 inputs?

### The design matrix

Every dataset used for regression can be written as a matrix `X`:

$$X = \begin{bmatrix} x_{11} & x_{12} & \dots & x_{1p} \\ x_{21} & x_{22} & \dots & x_{2p} \\ \vdots & \vdots & & \vdots \\ x_{n1} & x_{n2} & \dots & x_{np} \end{bmatrix}$$

```text
rows    = observations (one row per student)
columns = features     (one column per input variable)
```

With 100 students and 40 features: `X.shape = (100, 40)`. The model learns 40 coefficients plus one intercept — but **every observation still produces exactly one prediction**. That's the answer to "how can 30-40 variables produce one prediction?"

<details>
<summary>Show code</summary>

```python
rng = np.random.default_rng(seed=1)
n_students = 1000
n_features = 40

feature_names = ["hours_studied", "practice_problems", "attendance", "sleep_hours", "previous_score"]
feature_names += [f"feature_{i}" for i in range(6, n_features + 1)]

X_wide = rng.normal(size=(n_students, n_features))
true_weights = rng.normal(scale=2.0, size=n_features)
y_wide = 50 + X_wide @ true_weights + rng.normal(scale=5, size=n_students)

wide_model = LinearRegression()
wide_model.fit(X_wide, y_wide)

print("X.shape:            ", X_wide.shape)
print("model.coef_.shape:  ", wide_model.coef_.shape)

pd.DataFrame(X_wide[:5], columns=feature_names).round(2)
```

</details>

In [39]:
rng = np.random.default_rng(seed=1)
n_students = 1000
n_features = 40

feature_names = ["hours_studied", "practice_problems", "attendance", "sleep_hours", "previous_score"]
feature_names += [f"feature_{i}" for i in range(6, n_features + 1)]

X_wide = rng.normal(size=(n_students, n_features))
true_weights = rng.normal(scale=2.0, size=n_features)
y_wide = 50 + X_wide @ true_weights + rng.normal(scale=5, size=n_students)

wide_model = LinearRegression()
wide_model.fit(X_wide, y_wide)

print("X.shape:            ", X_wide.shape)
print("model.coef_.shape:  ", wide_model.coef_.shape)

pd.DataFrame(X_wide[:5], columns=feature_names).round(2)

X.shape:             (1000, 40)
model.coef_.shape:   (40,)


,hours_studied,practice_problems,attendance,sleep_hours,previous_score,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_31,feature_32,feature_33,feature_34,feature_35,feature_36,feature_37,feature_38,feature_39,feature_40
0,0.35,0.82,0.33,-1.30,0.91,0.45,-0.54,0.58,0.36,0.29,...,2.12,-1.11,-0.38,2.04,0.65,0.66,-0.51,-1.65,0.17,0.11
1,-1.23,-0.68,-0.07,-0.94,-0.10,0.10,0.04,-0.51,0.59,0.89,...,1.22,-0.30,-0.81,0.75,0.25,0.90,-0.35,-1.48,-0.11,-0.45
2,0.78,0.19,-1.63,-1.20,0.88,0.68,-0.64,-0.00,0.45,0.47,...,-0.69,0.14,-0.19,0.85,0.03,0.01,-0.71,0.47,-1.03,0.67
3,1.52,-1.52,-2.47,0.62,2.55,-1.00,-1.25,0.59,-0.84,-0.51,...,0.83,-0.59,-1.06,-0.90,-0.39,1.63,-1.18,0.16,-2.14,-0.00
4,0.90,-0.24,-0.63,0.23,0.70,0.66,1.97,0.21,-0.59,-0.13,...,-0.46,1.23,0.96,-2.71,0.04,-1.62,1.11,0.17,0.55,-1.07


### The mental model, for any number of features

```text
x1  * w1
x2  * w2
x3  * w3
...
x40 * w40
      ↓
     SUM
      +
  intercept
      ↓
  prediction
```

$$\hat y = b + w_1 x_1 + w_2 x_2 + \dots + w_{40} x_{40}$$

> Linear regression is called **linear** because these weighted feature contributions are added together linearly. The geometry becomes impossible for us to picture past 2 or 3 features, but the mathematical idea has not changed at all.

> ### 🧑‍🏫 Instructor note
>
> Misconception check: "More features means more outputs." Correct: many features can contribute to a
> single prediction — the model still outputs one number per observation, regardless of how many columns X has.

## Section 21 — Back to Two Features, Properly Named

Let's finish with our real 2-feature dataset and present the coefficients the way you'd actually want to read them — with names, not just a bare array.

<details>
<summary>Show code</summary>

```python
features = ["hours_studied", "practice_problems"]
X_final = scores_2d[features]
y_final = scores_2d["exam_score"]

final_model = LinearRegression()
final_model.fit(X_final, y_final)

coefficient_table = pd.DataFrame({
    "feature": ["Intercept"] + features,
    "coefficient": [final_model.intercept_] + list(final_model.coef_),
})
coefficient_table.round(2)
```

</details>

In [40]:
features = ["hours_studied", "practice_problems"]
X_final = scores_2d[features]
y_final = scores_2d["exam_score"]

final_model = LinearRegression()
final_model.fit(X_final, y_final)

coefficient_table = pd.DataFrame({
    "feature": ["Intercept"] + features,
    "coefficient": [final_model.intercept_] + list(final_model.coef_),
})
coefficient_table.round(2)

,feature,coefficient
0,Intercept,30.72
1,hours_studied,4.68
2,practice_problems,1.34


> Holding practice problems constant, one additional study hour is associated with approximately `w1` additional predicted points in this synthetic example. (We won't dig further into causal interpretation today.)

### One more prediction, with two features

In [41]:
new_hours_studied = 6
new_practice_problems = 10

manual_2d_prediction = (
    final_model.intercept_
    + final_model.coef_[0] * new_hours_studied
    + final_model.coef_[1] * new_practice_problems
)
sklearn_2d_prediction = final_model.predict([[new_hours_studied, new_practice_problems]])[0]

print(f"Manual prediction:  {manual_2d_prediction:.2f}")
print(f"sklearn prediction: {sklearn_2d_prediction:.2f}")

Manual prediction:  72.23
sklearn prediction: 72.23


/Users/abdukarimov/workspaces/work/humblebeeai/int.academy-tutorial/linear-regression/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Just like in 1D, let's see this new student sitting on the plane instead of just reading a number.

<details>
<summary>Show code</summary>

```python
new_prediction_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"), name="training students",
        hovertemplate="Hours: %{x}<br>Problems: %{y}<br>Score: %{z}<extra></extra>",
    )]
)
add_regression_plane(
    new_prediction_figure, scores_2d,
    final_model.intercept_, final_model.coef_[0], final_model.coef_[1],
)
new_prediction_figure.add_trace(
    go.Scatter3d(
        x=[new_hours_studied], y=[new_practice_problems], z=[sklearn_2d_prediction],
        mode="markers", marker=dict(size=7, color="#7c3aed", symbol="diamond"),
        name="new prediction (no actual value)",
        hovertemplate=(
            f"Hours: {new_hours_studied} (new/unseen)<br>Problems: {new_practice_problems}"
            f"<br>predicted score: {sklearn_2d_prediction:.1f}<extra></extra>"
        ),
    )
)
new_prediction_figure.update_layout(
    title="Predicting for a new student, shown on the plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
new_prediction_figure.show()
```

</details>

In [42]:
new_prediction_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"), name="training students",
        hovertemplate="Hours: %{x}<br>Problems: %{y}<br>Score: %{z}<extra></extra>",
    )]
)
add_regression_plane(
    new_prediction_figure, scores_2d,
    final_model.intercept_, final_model.coef_[0], final_model.coef_[1],
)
new_prediction_figure.add_trace(
    go.Scatter3d(
        x=[new_hours_studied], y=[new_practice_problems], z=[sklearn_2d_prediction],
        mode="markers", marker=dict(size=7, color="#7c3aed", symbol="diamond"),
        name="new prediction (no actual value)",
        hovertemplate=(
            f"Hours: {new_hours_studied} (new/unseen)<br>Problems: {new_practice_problems}"
            f"<br>predicted score: {sklearn_2d_prediction:.1f}<extra></extra>"
        ),
    )
)
new_prediction_figure.update_layout(
    title="Predicting for a new student, shown on the plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
new_prediction_figure.show()

A 30-feature version works **exactly the same way** — the library simply receives more columns.

## Recap

### One feature

$$\hat y = b + w_1 x_1$$

```text
LINE
```

### Two features

$$\hat y = b + w_1 x_1 + w_2 x_2$$

```text
PLANE
```

### Many features

$$\hat y = b + \sum_{j=1}^{p} w_j x_j$$

```text
HYPERPLANE
```

### The whole pipeline, one more time

```text
Prediction
    ↓
Residual
    ↓
Square
    ↓
Sum
    ↓
SSE
    ↓
Minimize
    ↓
Best coefficients
```

> Linear regression is not fundamentally about drawing a line. It is about learning coefficients that combine input features to produce predictions while minimizing prediction error. A line is simply the version we are lucky enough to visualize.

## Final Check

1. What is a residual?
2. Why don't we simply add raw residuals?
3. What does least squares try to minimize?
4. If one input produces a line, what do two inputs produce?
5. If a model has 40 input features, how can it still produce one prediction?

<details>
<summary>Show solution</summary>

```python
1. A residual is the difference between the actual observed value and the model's prediction: e_i = y_i - y_hat_i.
2. Raw residuals can be positive or negative and cancel out when summed, hiding how wrong the model actually is.
3. Least squares minimizes the sum of squared residuals (SSE) across all observations.
4. Two inputs produce a plane (in 3D); more generally, p inputs produce a hyperplane in (p+1)-dimensional space.
5. Each feature is multiplied by its learned coefficient, the contributions are added together with the
   intercept, and the result is one predicted value -- regardless of how many features go in.
```

</details>

---

*End of notebook. Continue exploring interactively in `streamlit_app/app.py`.*